# GRPO：PCB placement 品質最佳化 (genpcb)

從 **SFT 暖身後的 adapter** 接續，用 GRPO 最佳化 **routability**（用 surrogate 即時評分）。

- **reward**：Model A routability surrogate（GNN+CNN，grouped ρ 0.78，~ms/板）—— overlap/越界當硬 gate、HPWL 拿掉
- **group**：同 prompt(同 netlist) 內 `num_generations` 個 completion 做 group-relative advantage
- **起點**：SFT adapter（先跑 train_sft_colab.ipynb），GRPO 接續訓練它

> ✅ **這次最佳化的是真實可佈線性的代理（surrogate）**，不再是 HPWL proxy——HPWL 已證明會誤導（SA 壓 HPWL → Freerouting 卡死）。
> ⚠️ surrogate ρ 0.78 不完美 → 訓練後**必須用本機 Freerouting audit**（surrogate/audit.py）確認真實提升、抓 surrogate hacking；偏離就回灌重訓（DAgger）。
> ⚠️ KL 參考為 base（PEFT adapter-disable），beta 小，第一輪可接受。

> 執行前：Runtime → A100；Colab Secrets 設 `HF_TOKEN`、`GH_TOKEN`。

## 1. 確認 GPU

In [ ]:
!nvidia-smi

## 2. 安裝套件
需 TRL 含 `GRPOTrainer`（近期版本）。

In [ ]:
!pip -q install -U \
  "transformers>=5.0" "trl>=0.15" "peft>=0.13" \
  "bitsandbytes>=0.44" "accelerate>=1.0" "datasets>=3.0"

## 3. 設定（可直接改這格）

In [ ]:
MODEL           = "google/gemma-4-12b-it"          # 須與 SFT 同一顆
SFT_ADAPTER_DIR = "/content/drive/MyDrive/genpcb/qlora-gemma4-12b"   # train_sft_colab 的輸出
OUTPUT_DIR      = "/content/drive/MyDrive/genpcb/grpo-gemma4-12b"
N_PROMPTS       = 512        # 程序化 prompt 數（seed 與 SFT 錯開）
NUM_GENERATIONS = 8          # 每 prompt 生幾個 → group 大小
MAX_STEPS       = 80         # 先小規模確認 reward 會升（生成慢，200 步約 3.5hr）
LR              = 1e-6       # GRPO 遠小於 SFT
BETA            = 0.0        # KL=0：PEFT 參考是 base，beta>0 會把模型拉回沒學過格式的 base（上次崩潰主因）
TEMPERATURE     = 0.7        # 較低 → 取樣 completion 更常合法 → group 有真梯度
MAX_PROMPT_LEN  = 1024
MAX_COMPLETION_LEN = 512

## 4. Secrets + 掛 Drive + 取得 repo

In [ ]:
import os, sys
from google.colab import drive, userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")   # 有就開 W&B 即時看 reward 曲線
    USE_WANDB = True
except Exception:
    USE_WANDB = False
print("W&B:", "on" if USE_WANDB else "off（用訓練後的 reward 曲線 cell）")
drive.mount("/content/drive")
assert os.path.isdir(SFT_ADAPTER_DIR), f"找不到 SFT adapter：{SFT_ADAPTER_DIR}（先跑 train_sft_colab.ipynb）"

GH = userdata.get("GH_TOKEN")
if os.path.isdir("/content/genpcb"):
    !git -C /content/genpcb pull
else:
    !git clone https://{GH}@github.com/timmytsaa/genpcb.git /content/genpcb
!pip -q install -e /content/genpcb
sys.path.insert(0, "/content/genpcb/src")

## 5. Prompt 集 + reward 函式（皆 import 自 repo）
GRPO 自生 completion，不需標籤；prompt 為 placement 任務（板框+宣告+netlist+PLACE）。

In [ ]:
from datasets import Dataset
from genpcb.train.grpo import build_prompts, make_surrogate_reward_fn

prompt_ds = Dataset.from_list(build_prompts(N_PROMPTS, seed0=10000))   # 與 SFT(0..) 錯開
# routability surrogate 當即時 reward（grouped ρ 0.78；overlap/越界硬 gate、HPWL 拿掉）
reward_fn = make_surrogate_reward_fn("/content/genpcb/models/surrogate_a.pt")
print(len(prompt_ds), "prompts | reward = surrogate routability")
print(prompt_ds[0]["prompt"][:200])

## 6. 載入 base(4-bit) + SFT adapter（接續訓練）
adapter 設 `is_trainable=True`；GRPO 繼續訓練這個 LoRA。KL 參考為 base（adapter 關閉）。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, SFT_ADAPTER_DIR, is_trainable=True)
model.config.use_cache = False

In [ ]:
# 診斷：直接看模型實際吐什麼（出問題時先跑這格）
from genpcb.data.procedural import generate_board
from genpcb.data.serialize import board_to_dsl, dsl_to_sft_example
from genpcb.rewards import reward_from_completion

model.gradient_checkpointing_disable()   # 用 kv-cache，生成才快
model.config.use_cache = True
_ex = dsl_to_sft_example(board_to_dsl(generate_board("mcu", seed=20000)))
_inp = tok(_ex["prompt"], return_tensors="pt").to(model.device)
for _tag, _kw in [("greedy", dict(do_sample=False)),
                  ("sample@0.7", dict(do_sample=True, temperature=0.7))]:
    _out = model.generate(**_inp, max_new_tokens=384, **_kw)
    _gen = tok.decode(_out[0][_inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"===== {_tag}  reward={reward_from_completion(_ex['prompt'], _gen):.2f} =====")
    print(repr(_gen[:500]), "\n")

## 7. GRPO 訓練
生成是瓶頸（每 prompt 生 `NUM_GENERATIONS` 個，HF generate，無 vLLM）→ 煙霧規模 `MAX_STEPS` 先設小。

**看哪個指標**：GRPO 的 `loss` 會貼近 0、上下亂跳，**不是訓練信號**。要看 `reward`（平均，往上走才對）、`reward_std`、`kl`、`completion_length`——這些在 log 裡，下一格畫出來。

In [ ]:
from trl import GRPOConfig, GRPOTrainer

args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    beta=BETA,
    temperature=TEMPERATURE,
    num_generations=NUM_GENERATIONS,
    per_device_train_batch_size=NUM_GENERATIONS,   # 1 prompt 的 group / device step
    gradient_accumulation_steps=2,
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_COMPLETION_LEN,
    max_steps=MAX_STEPS,
    logging_steps=5,
    save_steps=40,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    use_vllm=False,
    report_to=("wandb" if USE_WANDB else "none"),   # 有 W&B 就即時看 reward
    run_name="grpo-gemma4-12b",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=args,
    train_dataset=prompt_ds,
    processing_class=tok,
)
trainer.train()

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

# GRPO 的真正訓練信號在 log_history（不是上面那張表的 loss）
df = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ["step", "loss", "reward", "reward_std", "kl", "completion_length"] if c in df]
print(df[cols].dropna(how="all").to_string())

if "reward" in df:
    d = df.dropna(subset=["reward"])
    fig, ax = plt.subplots(1, 2, figsize=(11, 3))
    ax[0].plot(d["step"], d["reward"]);        ax[0].set_title("reward（要往上）"); ax[0].set_xlabel("step")
    if "kl" in d: ax[1].plot(d["step"], d["kl"]); ax[1].set_title("kl（離 base 多遠）"); ax[1].set_xlabel("step")
    plt.tight_layout(); plt.show()
else:
    print("\n此 TRL 版本未在 log_history 記 reward；改開 W&B（設 WANDB_API_KEY）即時看。")

## 8. 存 adapter

In [ ]:
trainer.save_model(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)
print("GRPO adapter 已存到:", OUTPUT_DIR)

## 9. 評估：GRPO vs 程序化基線（surrogate routability）
held-out netlist 上比 GRPO 模型擺位 vs 程序化基線的 **surrogate 預測 routability**。
GRPO 應該把它推高。**但這是 surrogate 自評**——真實提升要靠下一格的本機 Freerouting audit。

In [ ]:
import numpy as np
from genpcb.data.procedural import FAMILIES, generate_board
from genpcb.data.serialize import board_to_dsl, dsl_to_sft_example
from genpcb.rewards import _board_from_completion
from genpcb.rewards.surrogate import load_surrogate, surrogate_routability

sm = load_surrogate("/content/genpcb/models/surrogate_a.pt")
model.gradient_checkpointing_disable()   # kv-cache，生成才快
model.config.use_cache = True
N_EVAL = 20
gen_rf, base_rf, valid, saved = [], [], 0, []
for i in range(N_EVAL):
    board = generate_board(FAMILIES[i % 3], seed=20000 + i)         # held-out
    ex = dsl_to_sft_example(board_to_dsl(board))
    inp = tok(ex["prompt"], return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=MAX_COMPLETION_LEN, do_sample=False)
    gen = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        b = _board_from_completion(ex["prompt"], gen)
        gen_rf.append(surrogate_routability(b, sm)); valid += 1
        saved.append((ex["prompt"], gen))          # 存給 audit 用
    except ValueError:
        gen_rf.append(0.0)
    base_rf.append(surrogate_routability(_board_from_completion(ex["prompt"], ex["completion"]), sm))

gen_rf, base_rf = np.array(gen_rf), np.array(base_rf)
print(f"合法率 {valid}/{N_EVAL}")
print(f"GRPO    surrogate routability: mean {gen_rf.mean():.3f}")
print(f"程序化基線 surrogate routability: mean {base_rf.mean():.3f}")
print(f"GRPO 勝過基線比例: {(gen_rf > base_rf).mean():.0%}")
# 存 GRPO 生成的 (prompt, completion) 給本機 Freerouting audit
import json
json.dump(saved, open("/content/drive/MyDrive/genpcb/grpo_rollouts.json", "w"))
print("已存 rollouts → Drive/grpo_rollouts.json（下載回本機跑 Freerouting audit）")

## 調參小抄 / 下一步

- **OOM**：降 `NUM_GENERATIONS`(8→4)、降 `MAX_COMPLETION_LEN`、`gradient_accumulation_steps=1`
- **reward 不升**：GRPO lr 太小/太大都會卡 → 試 5e-7 ~ 5e-6；或 group 太小(調大 `NUM_GENERATIONS`)
- **大量 -10（malformed）**：SFT 暖身不足 → 回頭加 SFT epoch/資料
- **太慢**：生成是瓶頸；穩定後改 `use_vllm=True`（需相容環境）

**成功標準**：訓練中 surrogate reward 平均上升；評估 cell GRPO routability > 程序化基線。

**關鍵下一步 = 本機 Freerouting audit（防 surrogate hacking）**：
下載 `grpo_rollouts.json` 回本機（有 Freerouting），跑：
```python
import json
from genpcb.rewards import _board_from_completion
from genpcb.rewards.surrogate import load_surrogate
from genpcb.surrogate.audit import audit, append_to_dataset
rollouts = json.load(open("grpo_rollouts.json"))
boards = [_board_from_completion(p, c) for p, c in rollouts]
m = load_surrogate("models/surrogate_a.pt")
rows, drift = audit(boards, m, jar="tools/freerouting.jar", k=12)
print(drift)   # audit_spearman 掉了 → surrogate 被鑽 → append_to_dataset 回灌重訓
```
若 GRPO 的 surrogate routability 升、但 audit 的真實 routed_fraction 沒升 → surrogate 被攻破，
回灌這些板重訓 surrogate（DAgger），再續訓 GRPO。